In [ ]:
import sys, os, subprocess
subprocess.run([sys.executable,'-m','pip','install','-q','onnxruntime'],check=False)
print('ort ready')


# BirdCLEF 2026 — Pre-compute BirdNET Embeddings (train soundscapes)

Extracts 1024-d BirdNET embeddings from the 66 training soundscapes.
Saves one `.npy` per 5-s window, named `birdnet_{sc_stem}_{end_secs}s.npy`,
matching the Perch embedding naming convention in `birdclef-2026-perch-embs-v3`.

For each 5-s window (e.g. 0–5 s), the **last 3 s** (2–5 s) are fed to BirdNET
(BirdNET expects 3-s clips at 48 kHz).

**Kaggle inputs required:**
1. `birdclef-2026`
2. BirdNET ONNX dataset (set `BIRDNET_ONNX_PATH` in Cell 3 if auto-detect fails)


In [ ]:
# === CELL 2: IMPORTS & CONFIG ===
import os, warnings
from pathlib import Path
import numpy as np, pandas as pd, soundfile as sf, librosa
import onnxruntime as ort
from tqdm import tqdm
warnings.filterwarnings('ignore')

CFG = dict(
    birdnet_sr      = 48000,
    birdnet_seconds = 3,
    birdnet_batch   = 32,
    # Perch windows are 5-s; we take the last 3-s of each for BirdNET
    perch_seconds   = 5,
)
CFG['birdnet_target'] = CFG['birdnet_sr'] * CFG['birdnet_seconds']  # 144000 samples
print(f"BirdNET target: {CFG['birdnet_target']} samples ({CFG['birdnet_seconds']}s @ {CFG['birdnet_sr']}Hz)")


In [ ]:
# === CELL 3: PATHS ===
def _fe(*c):
    return next((p for p in c if os.path.exists(p)), c[0])

TAXONOMY_CSV    = _fe('/kaggle/input/birdclef-2026/taxonomy.csv',
                      '/kaggle/input/competitions/birdclef-2026/taxonomy.csv')
SOUNDSCAPE_ANNO = _fe('/kaggle/input/birdclef-2026/train_soundscapes_labels.csv',
                      '/kaggle/input/competitions/birdclef-2026/train_soundscapes_labels.csv')
TRAIN_SC_DIR    = _fe('/kaggle/input/birdclef-2026/train_soundscapes',
                      '/kaggle/input/competitions/birdclef-2026/train_soundscapes')

# BirdNET ONNX — try common Kaggle dataset paths
BIRDNET_ONNX_PATH = None
for _c in [
    '/kaggle/input/birdnet-analyzer-onnx/BirdNET_GLOBAL_6K_V2.4_Model_FP32.onnx',
    '/kaggle/input/birdnet-analyzer/BirdNET_GLOBAL_6K_V2.4_Model_FP32.onnx',
    '/kaggle/input/birdnet-global-6k-v24/BirdNET_GLOBAL_6K_V2.4_Model_FP32.onnx',
    '/kaggle/input/birdnet-onnx/BirdNET_GLOBAL_6K_V2.4_Model_FP32.onnx',
    '/kaggle/input/birdnet/BirdNET_GLOBAL_6K_V2.4_Model_FP32.onnx',
]:
    if os.path.exists(_c):
        BIRDNET_ONNX_PATH = _c
        break

# *** If auto-detect fails, set path manually: ***
# BIRDNET_ONNX_PATH = '/kaggle/input/YOUR-DATASET/BirdNET_GLOBAL_6K_V2.4_Model_FP32.onnx'

OUT_DIR = Path('/kaggle/working/birdnet_embs')
OUT_DIR.mkdir(parents=True, exist_ok=True)

taxonomy_df = pd.read_csv(TAXONOMY_CSV)
species     = taxonomy_df['primary_label'].astype(str).tolist()
print(f'TRAIN_SC_DIR      : {TRAIN_SC_DIR}')
print(f'BIRDNET_ONNX_PATH : {BIRDNET_ONNX_PATH}')
print(f'OUT_DIR           : {OUT_DIR}')
sc_files = sorted(Path(TRAIN_SC_DIR).glob('*.ogg')) + sorted(Path(TRAIN_SC_DIR).glob('*.wav'))
print(f'Soundscape files  : {len(sc_files)}')
if BIRDNET_ONNX_PATH is None:
    print('\nWARNING: BirdNET ONNX not found. Attach the dataset and set BIRDNET_ONNX_PATH above.')


In [ ]:
# === CELL 4: BIRDNET ONNX SESSION + INSPECT ===
_sess = None; _inp = None; _eidx = None; _bnet_ok = False; _bnet_emb_dim = None

if BIRDNET_ONNX_PATH is None:
    print('ERROR: BIRDNET_ONNX_PATH is None -- check Cell 3')
else:
    try:
        opts = ort.SessionOptions()
        opts.graph_optimization_level = ort.GraphOptimizationLevel.ORT_ENABLE_ALL
        opts.intra_op_num_threads = os.cpu_count() or 4
        _sess = ort.InferenceSession(BIRDNET_ONNX_PATH, sess_options=opts,
                                     providers=['CPUExecutionProvider'])
        _inp  = _sess.get_inputs()[0].name
        _inp_shape = _sess.get_inputs()[0].shape
        print(f'Input  : {_inp}  shape={_inp_shape}')
        print(f'Outputs:')
        for i, o in enumerate(_sess.get_outputs()):
            print(f'  [{i}] {o.name}  shape={o.shape}')

        # Find embedding output: prefer 1024-d, else take last non-classifier output
        _out_names = [o.name for o in _sess.get_outputs()]
        _eidx = None
        for i, o in enumerate(_sess.get_outputs()):
            if o.shape and len(o.shape) >= 1 and o.shape[-1] == 1024:
                _eidx = i
                break
        if _eidx is None:
            # Fallback: pick the output with smallest last dim (embedding before classifier)
            candidates = [(i,o) for i,o in enumerate(_sess.get_outputs()) if o.shape]
            _eidx = min(candidates, key=lambda x: x[1].shape[-1])[0]

        # Dry run
        _dummy = np.zeros((1, CFG['birdnet_target']), np.float32)
        _out   = _sess.run(None, {_inp: _dummy})
        _emb   = _out[_eidx]
        if _emb.ndim == 3: _emb = _emb.mean(1)
        _bnet_emb_dim = _emb.shape[-1]
        _bnet_ok = True
        print(f'\nUsing output [{_eidx}]  emb_dim={_bnet_emb_dim}')
        print(f'BirdNET ONNX OK')
    except Exception as ex:
        print(f'BirdNET ONNX ERROR: {ex}')


In [ ]:
# === CELL 5: PARSE TRAINING SOUNDSCAPE WINDOW TIMESTAMPS ===
# We extract embeddings for the same windows used by Perch (every 5-s end)
def _parse_hms(s):
    p = str(s).strip().split(':')
    return int(p[0])*3600 + int(p[1])*60 + int(p[2])

sc_anno = pd.read_csv(SOUNDSCAPE_ANNO)
# Build per-soundscape window list: {sc_stem: [end_secs, ...]}
from collections import defaultdict
_sc_windows = defaultdict(set)
for _, row in sc_anno.iterrows():
    sc_stem  = Path(str(row['filename'])).stem
    end_secs = _parse_hms(row['end'])
    _sc_windows[sc_stem].add(end_secs)
_sc_windows = {k: sorted(v) for k, v in _sc_windows.items()}

total_windows = sum(len(v) for v in _sc_windows.values())
print(f'Soundscapes with labels : {len(_sc_windows)}')
print(f'Total windows           : {total_windows}')
print(f'Sample: {list(_sc_windows.items())[0]}')


In [ ]:
# === CELL 6: EMBED ALL SOUNDSCAPE WINDOWS ===
assert _bnet_ok, 'BirdNET ONNX not ready -- check Cell 4'

def birdnet_embed_windows(audio_path, end_secs_list):
    """For each 5-s window ending at end_secs, take the last 3-s and embed with BirdNET.
    Returns dict: {end_secs: emb (1024,)}
    """
    try:
        y, sr = sf.read(str(audio_path), always_2d=False)
    except Exception as e:
        return {}, str(e)
    if y.ndim == 2: y = y.mean(1)
    y = y.astype(np.float32)
    # Resample to BirdNET SR
    if sr != CFG['birdnet_sr']:
        y = librosa.resample(y, orig_sr=sr, target_sr=CFG['birdnet_sr'])

    clips = []
    for es in end_secs_list:
        # Take last 3-s of the 5-s window
        e0 = int(es * CFG['birdnet_sr'])
        s0 = max(0, e0 - CFG['birdnet_target'])
        c  = y[s0:e0]
        if len(c) < CFG['birdnet_target']:
            c = np.pad(c, (0, CFG['birdnet_target'] - len(c)))
        clips.append(c)

    # Batch inference
    all_embs = []
    for bi in range(0, len(clips), CFG['birdnet_batch']):
        B   = np.stack(clips[bi:bi+CFG['birdnet_batch']])
        out = _sess.run(None, {_inp: B})[_eidx]
        if out.ndim == 3: out = out.mean(1)
        all_embs.append(out.astype(np.float32))
    embs = np.vstack(all_embs)  # (N_windows, emb_dim)
    return dict(zip(end_secs_list, embs)), None


n_done = 0; n_skip = 0; n_err = 0

for sc_stem, end_secs_list in tqdm(_sc_windows.items(), desc='soundscapes'):
    # Find audio file
    ap = None
    for ext in ['.ogg', '.wav', '.flac']:
        c = Path(TRAIN_SC_DIR) / f'{sc_stem}{ext}'
        if c.exists():
            ap = c
            break
    if ap is None:
        n_err += 1
        continue

    emb_map, err = birdnet_embed_windows(ap, end_secs_list)
    if err:
        n_err += 1
        print(f'  ERR {sc_stem}: {err}')
        continue

    for es, emb in emb_map.items():
        out_path = OUT_DIR / f'birdnet_{sc_stem}_{es}s.npy'
        if out_path.exists():
            n_skip += 1
            continue
        np.save(str(out_path), emb)
        n_done += 1

all_npy = list(OUT_DIR.glob('birdnet_*.npy'))
print(f'Done: {n_done} saved  {n_skip} skipped  {n_err} errors')
print(f'Total .npy files: {len(all_npy)}')
if all_npy:
    s = np.load(str(all_npy[0]))
    print(f'Sample: {all_npy[0].name}  shape={s.shape}  emb_dim={_bnet_emb_dim}')


In [ ]:
# === CELL 7: UPLOAD AS birdclef-2026-birdnet-embs-v1 ===
import shutil, subprocess, json as _json

KAGGLE_USERNAME = os.environ.get('KAGGLE_USERNAME', 'chiragggg')
DATASET_SLUG    = 'birdclef-2026-birdnet-embs-v1'

_upload_dir = '/kaggle/working/upload_birdnet_embs'
os.makedirs(_upload_dir, exist_ok=True)

_copied = []
for npy in Path('/kaggle/working/birdnet_embs').glob('birdnet_*.npy'):
    shutil.copy2(str(npy), os.path.join(_upload_dir, npy.name))
    _copied.append(npy.name)
print(f'Files to upload: {len(_copied)}')

if not _copied:
    print('ERROR: no birdnet embeddings found')
else:
    _meta = {
        'title': DATASET_SLUG,
        'id': f'{KAGGLE_USERNAME}/{DATASET_SLUG}',
        'licenses': [{'name': 'CC0-1.0'}],
    }
    with open(os.path.join(_upload_dir, 'dataset-metadata.json'), 'w') as _mf:
        _json.dump(_meta, _mf, indent=2)
    _result = subprocess.run(
        ['kaggle','datasets','create','-p',_upload_dir,'--dir-mode','zip'],
        capture_output=True, text=True,
    )
    print(_result.stdout)
    if _result.returncode != 0:
        print('STDERR:', _result.stderr)
        print(f'If exists: kaggle datasets version -p {_upload_dir} -m "birdnet embs v1"')
    else:
        print(f'Upload complete: {KAGGLE_USERNAME}/{DATASET_SLUG}')
        print('emb_dim:', _bnet_emb_dim, '  windows:', len(_copied))
